# Question 19 - 58. Length of Last Word

Given a string `s` consisting of words and spaces, return *the length of the **last** word in the string*.

A **word** is a maximal substring consisting of non-space characters only.

**Example 1:**

    Input: s = "Hello World"
    Output: 5
    Explanation: The last word is "World" with length 5.

**Example 2:**

    Input: s = "   fly me   to   the moon  "
    Output: 4
    Explanation: The last word is "moon" with length 4.

**Example 3:**

    Input: s = "luffy is still joyboy"
    Output: 6
    Explanation: The last word is "joyboy" with length 6.

**Constraints:**

- `1 <= s.length <= 10^4`
- `s` consists of only English letters and spaces `' '`.
- There will be at least one word in `s`.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Scan from the end: skip trailing spaces, then count letters

📘 **Revise first:** [Part 5.4 · split vs split(' ')](./0-concepts-array-and-string.ipynb) · [Part 3.7 · looping backwards](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (split into words) | O(n) | O(n) |
| 2️⃣ Optimized (backward scan) | O(n) worst, usually much less | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Given a sentence, how long is its **last word**? A "word" is a run of non-space characters.

```
"Hello World"                → "World" → 5
"   fly me   to   the moon  " → "moon"  → 4   (note the trailing spaces!)
```

The trap is in Example 2: the sentence can **end with spaces**, and there can be **several spaces** between words.

**Everyday picture: reading a sentence backwards with your finger.** Put your finger on the **last character**.
1. While it's on a space, move left (skip the blank tail).
2. Now you're on the last letter of the last word. Move left, **counting**, until you hit a space or run out of sentence.

The count is the answer. You never even look at the beginning of the sentence.

### Step 0 · Clarify before coding

🗣️ *"I need the length of the last run of non-space characters. There may be leading, trailing, or multiple spaces, but there's always at least one word."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Trailing / multiple spaces? | **Yes.** | Must skip them, not count them. |
| At least one word guaranteed? | **Yes.** | The answer is never 0. |
| Only letters and spaces? | **Yes.** | No tabs or newlines to worry about. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** split the sentence into words and measure the last one.

Python's `s.split()` (with **no** argument) splits on any run of whitespace and throws away empty pieces, so `"  fly me  "` becomes `["fly", "me"]`.

🗣️ *"In Python the one-liner is len(s.split()[-1]). It's correct and O(n) time, but it builds a list of every word, O(n) extra memory, just to look at the last one."*

⚠️ **Gotcha worth saying out loud:** `s.split(" ")` (with an explicit space) does **not** collapse spaces. `"moon  ".split(" ")` gives `["moon", "", ""]`, so the "last word" would be the empty string `""`.

In [1]:
class SolutionBrute:
    def lengthOfLastWord(self, s: str) -> int:
        return len(s.split()[-1])

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** everything we need is at the **end** of the string. `split` processes the whole thing and builds every word, which is wasted effort. Read backwards instead, and stop as soon as the last word ends.

**Two simple loops, one index:**
1. Start `i` at the last character. **While `s[i]` is a space**, step left.
2. **While `s[i]` is not a space** (and we haven't run off the start), count it and step left.

🗣️ *"I'll scan from the end. First I skip trailing spaces, then I count characters until I hit a space or the start of the string. That uses O(1) extra space and only reads the tail of the string. In the worst case, one long word with no spaces, it reads all n characters, so it's O(n) time."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| `s.split()[-1]` | ✅ fine in practice | Readable and idiomatic, but O(n) memory and it processes the whole string. Expect "now without built-ins". |
| `s.rstrip().split(" ")[-1]` | ⚠️ | Works, but still allocates, and it's easy to forget the `rstrip()`. |
| Forward scan tracking "current word length" | ✅ | O(1) memory, but you must reset at spaces **and** remember the last non-zero length. More state to get wrong. |
| **Backward scan** | ✅ **best** | O(1) memory, reads only the tail, and needs no reset logic. |

In [2]:
class Solution:
    def lengthOfLastWord(self, s: str) -> int:
        i = len(s) - 1
        while i >= 0 and s[i] == " ":      # 1) skip the blank tail
            i -= 1
        length = 0
        while i >= 0 and s[i] != " ":      # 2) count the last word
            length += 1
            i -= 1
        return length

### Step 3 · Toy example walkthrough (tell it like a story)

`s = "fly me to the moon  "` (two trailing spaces, length 20, indexes 0–19)

| phase | i | s[i] | action | length |
|---|---|---|---|---|
| skip | 19 | space | step left | 0 |
| skip | 18 | space | step left | 0 |
| count | 17 | n | count | 1 |
| count | 16 | o | count | 2 |
| count | 15 | o | count | 3 |
| count | 14 | m | count | 4 |
| count | 13 | space | **stop** | **4** ✅ |

**Narrated:** "I start at the very end. It's a space, skip. Another space, skip. Now an 'n', the end of 'moon', so I start counting: n, o, o, m, that's four. Next is a space, so the word is over. The answer is 4, and I only looked at 7 of the 20 characters."

In [3]:
cases = [
    ("Hello World", 5),
    ("   fly me   to   the moon  ", 4),
    ("luffy is still joyboy", 6),
    ("a", 1),
    ("a ", 1),
    ("   abc", 3),
    ("day", 3),
]
for s, expected in cases:
    assert SolutionBrute().lengthOfLastWord(s) == expected
    assert Solution().lengthOfLastWord(s) == expected
    print(repr(s), "->", expected)
print("All tests passed ✅")

'Hello World' -> 5
'   fly me   to   the moon  ' -> 4
'luffy is still joyboy' -> 6
'a' -> 1
'a ' -> 1
'   abc' -> 3
'day' -> 3
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| `split()` | O(n) | O(n) | Breaks the whole sentence into a list of words. |
| **Backward scan** | **O(trailing spaces + last word)** ≤ O(n) | **O(1)** | Reads only the tail; stores one index and one counter. |

**Why at most O(n)?** Each character is visited at most once by one of the two loops. The worst case, a single long word, reads everything.

**Why O(1) memory?** Two integers, whatever the length.

**Edge cases to mention:** a single letter, trailing spaces, leading spaces, a single word.

**Likely follow-up:** *"What about tabs or newlines?"* → Use `s[i].isspace()` instead of `== " "`.

---

#### 🗣️ Full interview script (about 30 seconds)

*"I could use split and take the last word, but that builds a list of every word.*

*Instead I scan from the end: first skip any trailing spaces, then count characters until I hit a space or the start of the string.*

*That's O(1) extra space and reads only the end of the string. Worst case O(n) for one long word."*